# 🛡️ Projet : Prédiction de la Gravité des Accidents Routiers
> **Objectif :** Développer un modèle de classification capable d'identifier les accidents "Graves" (`1`) par rapport aux accidents "Non Graves" (`0`).
> 
> **Contexte :** Ce travail s'inscrit dans une démarche MLOps utilisant **MLflow** pour le suivi des expérimentations et la gestion du cycle de vie des modèles.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, log_loss
from sklearn.ensemble import GradientBoostingClassifier
import mlflow

## 1. Configuration de l'Environnement
Nous utilisons une base de donnée **SQLite** pour MLflow afin de garantir la persistance des données de tracking (paramètres, métriques et modèles).

In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [3]:
df_accident = pd.read_csv(r'C:\Users\Utilisateur\Documents\mlflow-EDUCATIONAL\data\dataset_accident.csv', sep=';' )

## 2. Préparation des Données
Le dataset est chargé depuis un fichier CSV. Nous effectuons une sélection de variables pour éviter le **Data Leakage** :
* **Cible (`y`)** : `grav_binary` (binaire).
* **Features (`X`)** : Toutes les colonnes sauf la cible et `grav_ordered` (qui contient une information sur le résultat final).

In [4]:
y = df_accident["grav_binary"]
X = df_accident.drop(columns=["grav_ordered", "grav_binary"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 📈 Phase 2 : Évaluation et Artefacts visuels

Dans cette phase, nous passons d'un simple tracking de chiffres à une analyse visuelle complète. Pour chaque modèle, nous générons désormais des outils de diagnostic essentiels :
* **Métadonnées du dataset** : Pour vérifier l'équilibre des classes.
* **Feature Importance** : Pour comprendre quelles variables (météo, vitesse, usagers) influencent le plus la gravité.
* **Matrice de Confusion** : Pour identifier précisément les erreurs de type "Faux Négatifs".
* **Courbes ROC**: Pour évaluer la capacité de séparation du modèle.

In [5]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

mlflow.set_experiment("Tuning")

configs = [
    {"n_estimators": 50,  "learning_rate": 0.3,  "max_depth": 2, "subsample": 1.0, "min_samples_split": 50},
    {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": 5, "subsample": 0.8, "min_samples_split": 10},
    {"n_estimators": 500, "learning_rate": 0.01, "max_depth": 9, "subsample": 0.5, "min_samples_split": 2},
]

# Métadonnées dataset
classes = np.unique(y_train)
class_ratio = {str(c): round(np.sum(y_train == c) / len(y_train), 3) for c in classes}

for config in configs:
    model = GradientBoostingClassifier(**config)
    run_name = f"GB_nest{config['n_estimators']}_depth{config['max_depth']}_mss{config['min_samples_split']}"

    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)

        # Params & métriques
        mlflow.log_params(model.get_params())
        mlflow.log_param("model_type", type(model).__name__)
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1_score": f1_score(y_test, y_pred, average='weighted'),
            "precision": precision_score(y_test, y_pred, average='weighted'),
            "recall": recall_score(y_test, y_pred, average='weighted'),
            "log_loss": log_loss(y_test, y_pred_proba)
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name=run_name)


        # ── Métadonnées dataset ──────────────────────────────────────
        mlflow.log_param("dataset_train_size", len(X_train))
        mlflow.log_param("dataset_test_size", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        for cls, ratio in class_ratio.items():
            mlflow.log_param(f"class_ratio_{cls}", ratio)

        # ── Feature importance ───────────────────────────────────────
        feature_importance = dict(zip(X_train.columns, model.feature_importances_))
        mlflow.log_params({f"feat_{k}": round(v, 4) for k, v in sorted(feature_importance.items(), key=lambda x: -x[1])})

        # ── Matrice de confusion ─────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
        ax.set_title(f"Confusion Matrix - {run_name}")
        plt.tight_layout()
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # ── Courbes ROC ──────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        
        if len(classes) == 2:
            # Cas binaire : une seule courbe
            fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        else:
            # Cas multiclasse
            y_test_bin = label_binarize(y_test, classes=classes)
            for i, cls in enumerate(classes):
                fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
                roc_auc = auc(fpr, tpr)
                ax.plot(fpr, tpr, label=f"Classe {cls} (AUC = {roc_auc:.2f})")
        
        ax.plot([0, 1], [0, 1], 'k--')
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"Courbes ROC - {run_name}")
        ax.legend()
        plt.tight_layout()
        plt.savefig("roc_curves.png")
        mlflow.log_artifact("roc_curves.png")
        plt.close()

2026/02/26 13:47:20 INFO mlflow.tracking.fluent: Experiment with name 'Tuning' does not exist. Creating a new experiment.
2026/02/26 13:47:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/26 13:48:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/02/26 13:51:12 WARNING mlflow.sklearn: Saving scikit-learn models in th

#  Phase 3 : Challenger le modèle avec XGBoost

Après avoir exploré le Gradient Boosting classique, nous introduisons **XGBoost** (*Extreme Gradient Boosting*). 
XGBoost est une implémentation optimisée du gradient boosting, reconnue pour sa rapidité et ses performances supérieures..

In [6]:
import xgboost as xgb
import mlflow.xgboost

with mlflow.start_run(run_name="XGBoost_baseline"):
    
    model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        eval_metric="logloss",
        random_state=42
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)

    mlflow.log_params(model.get_params())
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred, average='weighted'),
        "precision": precision_score(y_test, y_pred, average='weighted'),
        "recall": recall_score(y_test, y_pred, average='weighted'),
        "log_loss": log_loss(y_test, y_pred_proba)
    }
    mlflow.log_metrics(metrics)
    mlflow.xgboost.log_model(model, name="XGBoost_baseline")

    # ── Matrice de confusion ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    ax.set_title("Confusion Matrix - XGBoost_baseline")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifact("confusion_matrix.png")
    plt.close()

    # ── Courbes ROC ──────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba[:, 1])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    ax.plot([0, 1], [0, 1], 'k--')
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Courbes ROC - XGBoost_baseline")
    ax.legend()
    plt.tight_layout()
    plt.savefig("roc_curves.png")
    mlflow.log_artifact("roc_curves.png")
    plt.close()

In [7]:


configs_xgb = [
    {"name": "XGB_simple", "n_estimators": 50, "learning_rate": 0.3, "max_depth": 3},
    {"name": "XGB_balanced", "n_estimators": 150, "learning_rate": 0.1, "max_depth": 6},
    {"name": "XGB_agressive", "n_estimators": 500, "learning_rate": 0.01, "max_depth": 9}
]

for config in configs_xgb:
    current_config = config.copy()
    run_name = current_config.pop("name")
    
    with mlflow.start_run(run_name=run_name):
        model = xgb.XGBClassifier(
            **current_config,
            eval_metric="logloss",
            random_state=42
        )
        
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)
        
        current_recall = recall_score(y_test, y_pred, average='weighted')
        
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "recall": current_recall,
            "f1_score": f1_score(y_test, y_pred, average='weighted'),
            "precision": precision_score, average='weighted'),
            "log_loss": log_loss(y_test, y_pred_proba)
        }
        
        mlflow.log_params(current_config)
        mlflow.log_metrics(metrics)
        
        # Correction du Warning MLflow : Utilisation de name au lieu d'artifact_path si souhaité
        mlflow.xgboost.log_model(model, artifact_path="model")
        
        print(f"Run {run_name} terminé - Recall: {current_recall:.4f}")

SyntaxError: closing parenthesis ')' does not match opening parenthesis '{' on line 25 (3124589538.py, line 29)

## 3.1. Comparaison Globale
À ce stade, nous pouvons ouvrir l'interface MLflow pour comparer le **Recall** du modèle XGBoost avec celui du Gradient Boosting (Scikit-Learn). 

#  Phase 4 : Sélection Automatisée du Meilleur Modèle

Dans une approche **MLOps**, le choix du modèle ne doit pas être manuel. Nous utilisons ici l'API `MlflowClient` pour interroger notre base de données d'expériences et extraire le modèle le plus performant selon notre critère principal.

###  Critère de sélection : Le Recall
Pour notre problématique de sécurité routière, nous avons choisi de classer les modèles par **Recall décroissant**. Notre priorité absolue est de maximiser la détection des accidents graves, même si cela implique une légère baisse de précision globale.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Identification de l'expérience (Vérifie bien que le nom correspond au run précédent)
experiment_name = "XGBOOST-TEST-V2" 
experiment = client.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id

# Tri par Recall décroissant pour isoler le meilleur
runs = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.recall DESC"] 
)

if runs:
    best_run = runs[0]
    best_run_id = best_run.info.run_id # On récupère l'ID pour l'étape suivante
    print(f"🏆 Meilleur modèle : {best_run.info.run_name}")
    print(f"📈 Recall : {best_run.data.metrics['recall']:.4f}")
else:
    print("❌ Aucun run trouvé.")

## 5. Conclusion et Perspectives

Le modèle sélectionné peut maintenant être **enregistré dans le Model Registry** pour être servi via une API. 

**Bilan du projet :**
1. **Traçabilité** : Chaque test (Hyperparamètres, features) est historisé.
2. **Reproductibilité** : Le modèle et ses graphiques de performance sont sauvegardés sous forme d'artefacts.
3. **Optimisation Métier** : Le focus mis sur le Recall permet d'assurer une meilleure couverture des situations critiques sur la route.

**Évolutions possibles :** * Mise en place d'un monitoring de **Drifting** pour détecter si le comportement des accidents change avec le temps.
* Enrichissement du dataset avec des données externes (température précise, état de la chaussée).

## 5.1. Promotion du modèle "XGB_agressive" au Registre
Après analyse des performances, le modèle **XGB_agressive** a été retenu pour sa capacité supérieure à détecter les accidents graves (Recall maximal). 

Nous l'enregistrons officiellement dans le **Model Registry** sous un nom unique. Cela permet de centraliser les versions et de faciliter le futur déploiement.

In [ ]:
# Configuration du nom dans le registre
model_name_registry = "Detecteur_Accidents_Route"

if runs:
    # Construction de l'URI pointant directement vers l'artefact du run identifié
    model_uri = f"runs:/{best_run_id}/model"
    
    # Enregistrement officiel
    result = mlflow.register_model(model_uri, model_name_registry)
    
    print(f"✅ Modèle enregistré avec succès : {model_name_registry}")
else:
    print("⚠️ Action impossible : aucun meilleur run n'a été identifié.")

### 5.2. Transition de version
Une fois enregistré, le modèle apparaît dans l'onglet **Models** de l'UI MLflow. Nous pouvons alors lui attribuer un alias (ex: `Production`) pour indiquer aux applications utilisatrices que cette version est la version de référence validée.

# 🚀 Phase 6 : Déploiement et Utilisation du Modèle

Dans cette phase finale, nous simulons la mise en production du modèle. Plutôt que de recharger un fichier local, nous interrogeons le **Model Registry** de MLflow pour récupérer la version la plus récente (ou spécifique) du modèle validé.

Cela garantit que l'environnement qui utilise le modèle (par exemple, une application de secours routier) utilise exactement le même artefact que celui qui a été testé et validé précédemment.

In [ ]:
model = mlflow.xgboost.load_model(f"models:/Detecteur_Accidents_Route/latest")
print("Modèle chargé sans problème !")
print(f"Structure du modèle : \n{model}")

In [ ]:
import pandas as pd
import numpy as np

# Fixer la graine pour la reproductibilité
np.random.seed(42)
n_rows = 10

data = {
    'est_nuit': np.random.randint(0, 2, n_rows),
    'est_heure_pointe': np.random.randint(0, 2, n_rows),
    'jour_semaine': np.random.randint(1, 8, n_rows), # 1=Lundi, 7=Dimanche
    'est_weekend': np.random.randint(0, 2, n_rows),
    'agg': np.random.randint(0, 2, n_rows), # 1=En agglomération
    'vma': np.random.choice([30, 50, 70, 80, 110, 130], n_rows), # Vitesse Max Autorisée
    'impl_vehicule_leger': np.random.randint(0, 2, n_rows),
    'impl_poids_lourd': np.random.randint(0, 2, n_rows),
    'impl_pieton': np.random.randint(0, 2, n_rows),
}

df = pd.DataFrame(data)

# Logique simpliste pour la cible : grav_binary
# Plus de risque si : Nuit + Vitesse élevée + Poids lourd ou Piéton
score = (df['est_nuit'] * 2) + (df['vma'] / 50) + (df['impl_poids_lourd'] * 3) + (df['impl_pieton'] * 4)
df['grav_binary'] = (score > 6).astype(int)

# Réorganiser pour avoir la cible au début ou à la fin
cols = ['grav_binary'] + [c for c in df.columns if c != 'grav_binary']
df = df[cols]

df

In [ ]:
# On sépare X de la cible
X_new = df.drop(columns=['grav_binary'])

# Prédiction
predictions = model.predict(X_new)
df['predictions'] = predictions

print(df[['grav_binary', 'predictions']])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

y_pred = model.predict(X_new)
y_true = df['grav_binary']
print("--- Rapport de Performance (Nouveau Dataset) ---")
print(classification_report(y_true, y_pred))

# Focus sur le Recall pour la classe 1 (Accidents Graves)
rec = recall_score(y_true, y_pred)
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy (Détection Globale) : {acc:.2%}")
print(f"Recall (Détection Accidents Graves) : {rec:.2%}")